# ANTON — Notebook 04: Dataset Quality, Sources, and Training Readiness

Assesses quality grading breakdown, solver source provenance, and duplicate design clustering.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from anton.datasets.storage import DatasetStorageManager

storage = DatasetStorageManager()
stats = storage.db.get_statistics()

print("Quality Status Counts:")
for q, cnt in stats['quality_status_distribution'].items():
    print(f"  {q}: {cnt}")

print("
Data Source Counts:")
for s, cnt in stats['source_type_distribution'].items():
    print(f"  {s}: {cnt}")


In [ ]:
# Duplicate and near-duplicate audit
from anton.datasets.validation.duplicate_detector import DuplicateDetector
from anton.datasets.schema import DatasetRecord

records_meta = storage.db.list_records(limit=500)
records = []
for m in records_meta:
    raw = storage.get_record(m['record_id'])
    if raw:
        records.append(DatasetRecord.model_validate(raw))

if records:
    detector = DuplicateDetector(relative_param_tolerance=0.01)
    dup_report = detector.analyze_dataset_duplicates(records)
    print("=== Duplicate Geometry Audit ===")
    print(f"Total Records: {dup_report['total_records_analyzed']}")
    print(f"Unique Geometries: {dup_report['unique_geometries_count']}")
    print(f"Near Duplicate Pairs: {len(dup_report['near_duplicate_pairs'])}")
else:
    print("[EMPTY STATE] No records available to audit duplicates.")
